In [43]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "data":
    project_root = project_root.parent

parquet_path = project_root / "data" / "sentiment_absa.parquet"
df = pd.read_parquet(parquet_path)

# Conversione in datetime
df['created_utc'] = pd.to_datetime(df['created_utc'], unit='s')

# RIMUOVE la timezone per rendere la colonna compatibile con Excel
df['created_utc'] = df['created_utc'].dt.tz_localize(None)


#df = (df['full_text'] != df['clean_text'])
df = df[(df['created_utc'] > "2021-06-01") & (df['created_utc'] < "2022-06-30")]
df['sentiment_label'].value_counts()

sentiment_label
neutral     5751
negative    4316
positive    3181
Name: count, dtype: int64

In [ ]:
import plotly.graph_objects as go

lang_counts = df['lang'].value_counts()

fig = go.Figure(data=[go.Pie(
    labels=lang_counts.index,
    values=lang_counts.values,
    hole=0.45, 
    marker=dict(
        colors=["#ff7f0e", "#d2701a", "#ffcf0e", "#ffdb70"],
        line=dict(color='#ffffff', width=2)  
    ),
    textinfo='label+percent',      
    textposition='inside',         
    insidetextorientation='radial' 
)])

# 3. Impostazioni di Layout
fig.update_layout(
    title=dict(
        text="Language Distribution",
        x=0.5, # Centra il titolo
        font=dict(size=20)
    ),
    template="plotly_white",
    font=dict(size=18),
    showlegend=False, 
    margin=dict(t=80, b=40, l=40, r=40),
    width=800,
    height=600
)

fig.show()

In [26]:
df.describe()

,score
count,1.153107e+06
mean,9.619768e+00
std,2.690262e+01
min,-4.000000e+00
25%,1.000000e+00
50%,3.000000e+00
75%,8.000000e+00
max,4.459000e+03


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


df["created_utc"] = pd.to_datetime(df["created_utc"], utc=True)


df["content_type"] = df["doc_id"].str[:2].map({
    "t3": "Post",
    "t1": "Comment"
})

conditions = [
    (df["created_utc"] >= pd.Timestamp("2021-08-30", tz="UTC")) & (df["created_utc"] <= pd.Timestamp("2022-05-30", tz="UTC")),
    (df["created_utc"] >= pd.Timestamp("2022-08-30", tz="UTC")) & (df["created_utc"] <= pd.Timestamp("2023-05-30", tz="UTC")),
    (df["created_utc"] >= pd.Timestamp("2023-08-30", tz="UTC")) & (df["created_utc"] <= pd.Timestamp("2024-05-30", tz="UTC")),
    (df["created_utc"] >= pd.Timestamp("2024-08-30", tz="UTC")) & (df["created_utc"] <= pd.Timestamp("2025-05-30", tz="UTC")),
    (df["created_utc"] >= pd.Timestamp("2025-08-30", tz="UTC")) & (df["created_utc"] <= pd.Timestamp("2026-05-30", tz="UTC"))
]


choices = ["21/22", "22/23", "23/24", "24/25", "25/26"]


df["season"] = np.select(conditions, choices, default=None)


df_season = df[df["season"] != "None"]

counts = (
    df_season.groupby(["season", "content_type"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

# 4. Plot
fig = go.Figure()

fig.add_bar(
    x=counts.index,
    y=counts["Post"],
    name="Posts",
    text=counts["Post"],
    textposition="auto",    
    marker_color="#ffcf0e"
)

fig.add_bar(
    x=counts.index,
    y=counts["Comment"],
    name="Comments",
    text=counts["Comment"],
    textposition="auto",   
    marker_color="#d2701a"
)

fig.update_layout(
    barmode="group",       
    template="plotly_white",
    font=dict(size=16),
    width=1000,
    height=600
)

fig.show()

In [10]:
parquet_path = project_root / "data" / "raw_2021-22.parquet"
df = pd.read_parquet(parquet_path)
df

,id,type,created_utc,author,subreddit,title,body,score,permalink,parent_id
0,t3_pe7goe,post,2021-08-30 00:39:18+00:00,Munfury,ACMilan,Sigma males,,85,https://www.reddit.com/r/ACMilan/comments/pe7g...,NaN
1,t3_pe8jta,post,2021-08-30 01:49:12+00:00,matheusagra,ACMilan,found this 2007 ball in my basement,,83,https://www.reddit.com/r/ACMilan/comments/pe8j...,NaN
2,t3_peabod,post,2021-08-30 03:46:05+00:00,arshadshabick,ACMilan,More post in the stadium,The possibility of me ever visiting san siro i...,38,https://www.reddit.com/r/ACMilan/comments/peab...,NaN
3,t3_peac2q,post,2021-08-30 03:46:53+00:00,Hass_s,ACMilan,Lorenzo Colombo scored this beauty last night,,312,https://www.reddit.com/r/ACMilan/comments/peac...,NaN
4,t3_pebb98,post,2021-08-30 04:55:44+00:00,vanadial,ACMilan,Old Match Replay,Anyone know where can I watch AC Milan old ful...,18,https://www.reddit.com/r/ACMilan/comments/pebb...,NaN
...,...,...,...,...,...,...,...,...,...,...
225564,t1_ie94tit,comment,2022-06-29 23:47:34+00:00,I_hate_abbrev,ACMilan,,* Divin Codino,1,https://www.reddit.com/r/ACMilan/comments/vnnl...,t3_vnnl3g
225565,t1_ie94zye,comment,2022-06-29 23:48:56+00:00,HommoFroggy,ACMilan,,According to what has been reported the issue ...,2,https://www.reddit.com/r/ACMilan/comments/vnt1...,t3_vnt1cy
225566,t1_ie958zh,comment,2022-06-29 23:50:49+00:00,fasobelli,ACMilan,,No idea on either. For the first time I can’t...,1,https://www.reddit.com/r/ACMilan/comments/vnro...,t3_vnropr
225567,t1_ie969fd,comment,2022-06-29 23:58:39+00:00,piedraa,ACMilan,,As he should. I think Gazidis is only here unt...,5,https://www.reddit.com/r/ACMilan/comments/vnt1...,t3_vnt1cy


In [7]:
df_commenti = df[df["type"] == "comment"]
df_commenti

,id,type,created_utc,author,subreddit,title,body,score,permalink,parent_id,full_text,clean_text,lang
914,t1_iig2xyh,comment,2022-08-01 00:03:05+00:00,TheSpartanLion,ACMilan,,And that's why we love him,9,https://www.reddit.com/r/ACMilan/comments/wcsy...,t3_wcsyod,And that's why we love him,And that's why we love him,en
915,t1_iig4kcv,comment,2022-08-01 00:16:16+00:00,yllimameni,ACMilan,,>Both could arrive.,0,https://www.reddit.com/r/ACMilan/comments/wd0d...,t3_wd0dnn,>Both could arrive.,>Both could arrive.,en
916,t1_iig582e,comment,2022-08-01 00:21:34+00:00,MeanMikeMaignan,ACMilan,,"Heade 1-2 mistakes but was ok after that, had ...",2,https://www.reddit.com/r/ACMilan/comments/wcuo...,t3_wcuokq,"Heade 1-2 mistakes but was ok after that, had ...","Heade 1-2 mistakes but was ok after that, had ...",en
917,t1_iig5976,comment,2022-08-01 00:21:49+00:00,CurryMuncher78,ACMilan,,And we are still looking for one. He was a las...,23,https://www.reddit.com/r/ACMilan/comments/wd0d...,t3_wd0dnn,And we are still looking for one. He was a las...,And we are still looking for one. He was a las...,en
918,t1_iig5sdo,comment,2022-08-01 00:26:12+00:00,fasobelli,ACMilan,,All he said is that somebody thinks both are c...,0,https://www.reddit.com/r/ACMilan/comments/wd0d...,t3_wd0dnn,All he said is that somebody thinks both are c...,All he said is that somebody thinks both are c...,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1153102,t1_op179ex,comment,2026-05-31 23:29:26+00:00,dukesdj,ACMilan,,Have you checked out AC Milan Zone? They are d...,6,https://www.reddit.com/r/ACMilan/comments/1tt8...,t3_1tt8dib,Have you checked out AC Milan Zone? They are d...,Have you checked out AC Milan Zone? They are d...,en
1153103,t1_op19r2a,comment,2026-05-31 23:44:13+00:00,redbirdsucks,ACMilan,,Allegri is officially on the terrorist watchlist,3,https://www.reddit.com/r/ACMilan/comments/1tt5...,t3_1tt5r4b,Allegri is officially on the terrorist watchlist,Allegri is officially on the terrorist watchlist,en
1153104,t1_op1a0oc,comment,2026-05-31 23:45:48+00:00,Ciccio_Camarda,ACMilan,,> Theres no space in todays game for wingers l...,1,https://www.reddit.com/r/ACMilan/comments/1tt8...,t3_1tt8dib,> Theres no space in todays game for wingers l...,> Theres no space in todays game for wingers l...,en
1153105,t1_op1aw0r,comment,2026-05-31 23:50:55+00:00,ATLfalcons27,ACMilan,,Luckily only the people who have always dislik...,3,https://www.reddit.com/r/ACMilan/comments/1tt5...,t3_1tt5r4b,Luckily only the people who have always dislik...,Luckily only the people who have always dislik...,en


In [10]:
parquet_path = project_root / "data" / "sentiment_absa.parquet"
df = pd.read_parquet(parquet_path)
df[['sentence', 'aspect_target', 'sentiment_label']]

,sentence,aspect_target,sentiment_label
0,Zlatan watching the game today,Ibrahimovic,neutral
1,Maldini has left Casa Milan after concluding t...,Maldini,negative
2,Maldini has definitely been through a lot fini...,Maldini,negative
3,What Maldini thinking here?,Maldini,neutral
4,Most expensive signings under Maldini and Masa...,Maldini,negative
...,...,...,...
1075,Now that the RedBird deal is finalized more fu...,Cardinale,neutral
1076,Maldini and co need to resign if they let Leao...,Maldini,negative
1077,Watch Inter demolish Cremonese 5-0 While Roma ...,Maldini,positive
1078,the fact that nobody passes to giroud is getti...,Ibrahimovic,negative


In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "data":
    project_root = project_root.parent

parquet_path = project_root / "data" / "clean_text.parquet"
df = pd.read_parquet(parquet_path)

df['created_utc'] = pd.to_datetime(df['created_utc'], unit='s')

df['created_utc'] = df['created_utc'].dt.tz_localize(None)
df


,id,type,created_utc,author,subreddit,title,body,score,permalink,parent_id,full_text,clean_text,lang
0,t3_wd7rxu,post,2022-08-01 04:05:36,travis-bickle06,ACMilan,A Little Edit I Made Yesterday😅 (ib:@rmvidstv ...,,151,https://www.reddit.com/r/ACMilan/comments/wd7r...,NaN,A Little Edit I Made Yesterday😅 (ib:@rmvidstv ...,A Little Edit I Made Yesterday😅 (ib:@rmvidstv ...,en
1,t3_wd85zj,post,2022-08-01 04:26:53,ACMilang,ACMilan,Zlatan watching the game today,,203,https://www.reddit.com/r/ACMilan/comments/wd85...,NaN,Zlatan watching the game today,Zlatan watching the game today,en
2,t3_wdbpbd,post,2022-08-01 08:00:15,mercurialsaliva,ACMilan,Weekly Discussion Thread,"This is where your thoughts, questions, compar...",28,https://www.reddit.com/r/ACMilan/comments/wdbp...,NaN,Weekly Discussion Thread This is where your th...,Weekly Discussion Thread This is where your th...,en
3,t3_wdc9qp,post,2022-08-01 08:37:48,MilesOfPebbles,ACMilan,The 2022/23 Coppa Italia bracket is set after ...,,56,https://www.reddit.com/r/ACMilan/comments/wdc9...,NaN,The 2022/23 Coppa Italia bracket is set after ...,The 2022/23 Coppa Italia bracket is set after ...,en
4,t3_wdda38,post,2022-08-01 09:42:42,Hass_s,ACMilan,One from the archives 🇧🇪,,361,https://www.reddit.com/r/ACMilan/comments/wdda...,NaN,One from the archives 🇧🇪,One from the archives 🇧🇪,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1153102,t1_op179ex,comment,2026-05-31 23:29:26,dukesdj,ACMilan,,Have you checked out AC Milan Zone? They are d...,6,https://www.reddit.com/r/ACMilan/comments/1tt8...,t3_1tt8dib,Have you checked out AC Milan Zone? They are d...,Have you checked out AC Milan Zone? They are d...,en
1153103,t1_op19r2a,comment,2026-05-31 23:44:13,redbirdsucks,ACMilan,,Allegri is officially on the terrorist watchlist,3,https://www.reddit.com/r/ACMilan/comments/1tt5...,t3_1tt5r4b,Allegri is officially on the terrorist watchlist,Allegri is officially on the terrorist watchlist,en
1153104,t1_op1a0oc,comment,2026-05-31 23:45:48,Ciccio_Camarda,ACMilan,,> Theres no space in todays game for wingers l...,1,https://www.reddit.com/r/ACMilan/comments/1tt8...,t3_1tt8dib,> Theres no space in todays game for wingers l...,> Theres no space in todays game for wingers l...,en
1153105,t1_op1aw0r,comment,2026-05-31 23:50:55,ATLfalcons27,ACMilan,,Luckily only the people who have always dislik...,3,https://www.reddit.com/r/ACMilan/comments/1tt5...,t3_1tt5r4b,Luckily only the people who have always dislik...,Luckily only the people who have always dislik...,en


In [5]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "data":
    project_root = project_root.parent

parquet_path = project_root / "data" / "sentiment_absa.parquet"
df = pd.read_parquet(parquet_path)

df['created_utc'] = pd.to_datetime(df['created_utc'], unit='s')

df['created_utc'] = df['created_utc'].dt.tz_localize(None)
df = df[df['n_executives']>1]
df.head()


,doc_id,sentence_id,created_utc,lang,score,sentence,executive,n_executives,sentence_en,aspect_target,sentiment_label,p_neg,p_neu,p_pos,sentiment_score
11,t3_wgomfo,t3_wgomfo#s48,2022-08-05 07:04:57,en,186,It was already agreed that the philosophy woul...,maldini,2,It was already agreed that the philosophy woul...,Maldini,neutral,0.148010,0.838614,0.013376,-0.134634
12,t3_wgomfo,t3_wgomfo#s48,2022-08-05 07:04:57,en,186,It was already agreed that the philosophy woul...,massara,2,It was already agreed that the philosophy woul...,Massara,neutral,0.128390,0.857522,0.014088,-0.114302
14,t3_wgomfo,t3_wgomfo#s54,2022-08-05 07:04:57,en,186,"It works the same way, my references are Paolo...",gazidis,2,"It works the same way, my references are Paolo...",Gazidis,positive,0.033349,0.380150,0.586502,0.553153
15,t3_wgomfo,t3_wgomfo#s54,2022-08-05 07:04:57,en,186,"It works the same way, my references are Paolo...",maldini,2,"It works the same way, my references are Paolo...",Maldini,positive,0.038815,0.337329,0.623856,0.585041
26,t3_wirda4,t3_wirda4#s0,2022-08-07 22:00:34,en,0,Unpopular opinion: Massara and Maldini are doi...,maldini,2,Unpopular opinion: Massara and Maldini are doi...,Maldini,negative,0.980443,0.016033,0.003524,-0.976919


In [7]:
df['executive'].unique()

<ArrowStringArray>
[    'maldini',     'massara',     'gazidis', 'ibrahimovic',   'cardinale',
     'moncada',     'furlani']
Length: 7, dtype: str